# Exercise 8-1: Analyze the NSF Awards data

## Read the data

In [36]:
import pandas as pd
import datetime

data_dir = '../../data'

In [37]:
awards_df = pd.read_pickle(f'{data_dir}/pkl_files/awards_data_2010_2025.pkl')

### Reshape the Data Frame for this exercise

In [38]:
# Change date columns to datetime
awards_df = awards_df.astype({'awd_eff_date': 'datetime64[ns]', 'awd_exp_date': 'datetime64[ns]', 'awd_min_amd_letter_date': 'datetime64[ns]', 'awd_max_amd_letter_date': 'datetime64[ns]'})

In [39]:
# Add awd_month and awd_year columns
awards_df['awd_monthnum'] = awards_df['awd_eff_date'].dt.month
awards_df['awd_month'] = awards_df['awd_eff_date'].dt.month_name()
awards_df['awd_year'] = awards_df['awd_eff_date'].dt.year

In [40]:
# Get rid of most of the columns
awards_df = awards_df[['dir_abbr', 'awd_year', 'awd_monthnum', 'awd_month', 'awd_id', 'awd_amount']]
# Add a column for the number of awards
awards_df['awd_count'] = 1

In [41]:
awards_df.head()

,dir_abbr,awd_year,awd_monthnum,awd_month,awd_id,awd_amount,awd_count
0,MPS,2010,1,January,0415302,146000000.0,1
1,EDU,2009,12,December,0731599,10000.0,1
2,MPS,2010,9,September,0804541,155998.0,1
3,MPS,2010,4,April,0805878,196001.0,1
4,MPS,2010,9,September,0805989,196252.0,1


## Group and aggregate the data

In [ ]:
# Group the awards_df DataFrame by 'dir_abbr' and 'awd_year' columns
# This will allow us to perform aggregation operations (e.g., sum, mean) on the grouped data
awards_grouped = awards_df.groupby(['dir_abbr', 'awd_year'])

In [43]:
# Aggregate the grouped data by summing up the numeric columns
awards_by_year = awards_grouped.sum()
awards_by_year.head(3)

awd_monthnum  \
dir_abbr awd_year                 
BFA      2010                72   
         2011                38   
         2012                30   

                                                           awd_month  \
dir_abbr awd_year                                                      
BFA      2010      JanuaryJanuaryMarchMayMayJuneJuneAugustAugustS...   
         2011                       JuneJuneAugustSeptemberSeptember   
         2012                                    MayJuneJuneJuneJuly   

                                                              awd_id  \
dir_abbr awd_year                                                      
BFA      2010      1019712101995410355491037979103957510417941043...   
         2011                    11399011140249115293911567841157476   
         2012                    12406011242711124298212448561247855   

                   awd_amount  awd_count  
dir_abbr awd_year                         
BFA      2010       3350885.0         12  
         2011        559879.0          5  
         2012       1317168.0          5

In [44]:
# Drop unnecessary columns to simplify the DataFrame
awards_by_year.drop(columns=['awd_month', 'awd_monthnum', 'awd_id'], inplace=True)
awards_by_year.reset_index(inplace=True)
awards_by_year.head(3)

,dir_abbr,awd_year,awd_amount,awd_count
0,BFA,2010,3350885.0,12
1,BFA,2011,559879.0,5
2,BFA,2012,1317168.0,5


## Use pivot tables

In [45]:
# Filter the awards_by_year DataFrame to include only rows where awd_year is greater than or equal to 2022
# and reset the index for the resulting DataFrame
awards_recent = awards_by_year.query('awd_year >= 2022').reset_index()
awards_recent.head(5)

,index,dir_abbr,awd_year,awd_amount,awd_count
0,12,BFA,2022,2377470.0,13
1,26,BIO,2022,763007359.0,1198
2,27,BIO,2023,839596545.0,1115
3,28,BIO,2024,595357831.0,1018
4,29,BIO,2025,122278582.0,272


In [46]:
# Create a pivot table to display the award amounts by directorate (dir_abbr) and year (awd_year)
awards_recent.pivot(index='dir_abbr', columns='awd_year', values='awd_amount').head()

awd_year,2022,2023,2024,2025,2026
dir_abbr,,,,,
BFA,2.377470e+06,NaN,NaN,NaN,NaN
BIO,7.630074e+08,8.395965e+08,595357831.0,122278582.0,2160000.0
CSE,1.051493e+09,1.054025e+09,805756689.0,155694943.0,600000.0
EDU,1.155589e+09,8.172164e+08,838175584.0,84790865.0,NaN
ENG,7.888189e+08,7.123550e+08,659459856.0,111612106.0,NaN


In [47]:
# Create a pivot table to display the number of awards (awd_count) by directorate (dir_abbr) and year (awd_year) for years >= 2022
awards_df.query('awd_year >= 2022').pivot_table(
    index='dir_abbr', columns='awd_year', values='awd_count', aggfunc='sum').head()

awd_year,2022,2023,2024,2025,2026
dir_abbr,,,,,
BFA,13.0,NaN,NaN,NaN,NaN
BIO,1198.0,1115.0,1018.0,272.0,8.0
CSE,2027.0,2115.0,1724.0,367.0,1.0
EDU,1211.0,1152.0,1314.0,113.0,NaN
ENG,1721.0,1689.0,1628.0,293.0,NaN


## Work with bins

In [48]:
# Reset the index of the awards_by_year DataFrame to make it easier to work with
awards_by_year.reset_index(inplace=True)
awards_by_year

,index,dir_abbr,awd_year,awd_amount,awd_count
0,0,BFA,2010,3350885.0,12
1,1,BFA,2011,559879.0,5
2,2,BFA,2012,1317168.0,5
3,3,BFA,2013,946559.0,4
4,4,BFA,2014,3014470.0,6
...,...,...,...,...,...
208,208,TIP,2021,366452493.0,904
209,209,TIP,2022,429729426.0,804
210,210,TIP,2023,582735986.0,1000
211,211,TIP,2024,640899526.0,990


In [ ]:
# Create a new column 'decade' in the awards_by_year DataFrame by binning the 'awd_year' column
# into two categories: '2010s' and '2020s', based on specified year ranges
awards_by_year['decade'] = pd.cut(awards_by_year.awd_year, 
                                 bins=[2010,2020,2030], 
                                 labels=['2010s', '2020s'], 
                                 right=False)

In [50]:
awards_by_year.head(25)

,index,dir_abbr,awd_year,awd_amount,awd_count,decade
0,0,BFA,2010,3.350885e+06,12,2010s
1,1,BFA,2011,5.598790e+05,5,2010s
2,2,BFA,2012,1.317168e+06,5,2010s
3,3,BFA,2013,9.465590e+05,4,2010s
4,4,BFA,2014,3.014470e+06,6,2010s
5,5,BFA,2015,8.356830e+05,4,2010s
6,6,BFA,2016,1.126021e+06,8,2010s
7,7,BFA,2017,8.623370e+05,8,2010s
8,8,BFA,2018,2.300890e+06,11,2010s
9,9,BFA,2019,3.274493e+06,11,2010s


In [51]:
# Drop the 'awd_year' column from the awards_by_year DataFrame to prepare for grouping by decade
awards_by_decade = awards_by_year.drop(columns=['awd_year'])
awards_by_decade.head()

,index,dir_abbr,awd_amount,awd_count,decade
0,0,BFA,3350885.0,12,2010s
1,1,BFA,559879.0,5,2010s
2,2,BFA,1317168.0,5,2010s
3,3,BFA,946559.0,4,2010s
4,4,BFA,3014470.0,6,2010s


In [52]:
# Group the awards_by_decade DataFrame by 'dir_abbr' and 'decade' columns and sum the numeric columns
awards_by_decade = awards_by_decade.groupby(['dir_abbr', 'decade']).sum()  
awards_by_decade.reset_index(inplace=True)
awards_by_decade.head(25)

C:\Users\dan\AppData\Local\Temp\ipykernel_49788\3826738566.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  awards_by_decade = awards_by_decade.groupby(['dir_abbr','decade']).sum()


,dir_abbr,decade,index,awd_amount,awd_count
0,BFA,2010s,45,1.758838e+07,74
1,BFA,2020s,33,6.430475e+06,34
2,BIO,2010s,185,7.875422e+09,13325
3,BIO,2020s,189,4.004254e+09,6235
4,CSE,2010s,375,9.414157e+09,19239
5,CSE,2020s,322,5.275407e+09,10255
6,EDU,2010s,555,1.122537e+10,10564
7,EDU,2020s,381,5.632134e+09,6175
8,ENG,2010s,735,6.640349e+09,18122
9,ENG,2020s,489,4.089436e+09,8650
